In [1]:
import json
import os
from typing import Literal

import fast_rlm
import networkx as nx
from datasets import load_dataset
from fast_rlm import RLMConfig
from pydantic import BaseModel

DATASET_NAME = "rmanluo/RoG-webqsp"
MODEL = os.getenv("RLM_MODEL", "minimax-m3:cloud")
os.environ.setdefault("RLM_MODEL_BASE_URL", "http://localhost:11434/v1")
os.environ.setdefault("RLM_MODEL_API_KEY", "ollama")

/Users/kapish/Kapish/RP/fast-rlm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'ollama'

In [15]:
test_rows = load_dataset(DATASET_NAME, split="test").select(range(50))
example = test_rows[0]

combined_graph = nx.MultiDiGraph()
seen_triples = set()
for row in test_rows:
    for subject, relation, object_ in row["graph"]:
        triple = (str(subject), str(relation), str(object_))
        if triple in seen_triples:
            continue
        seen_triples.add(triple)
        combined_graph.add_edge(
            triple[0],
            triple[2],
            key=triple[1],
            relation=triple[1],
        )
indexed_graph = fast_rlm.prepare_graph(combined_graph)

In [16]:
print(len(test_rows))

50


In [3]:
class WebQSPAnswer(BaseModel):
    answer_entities: list[str]
    status: Literal["answered", "insufficient_data"]
    message: str


config = RLMConfig(
    primary_agent=MODEL,
    sub_agent=MODEL,
    max_depth=1,
    max_calls_per_subagent=4,
    max_global_calls=20,
    max_graph_queries=1,
    max_graph_hops=4,
    min_entity_confidence=0.75,
    truncate_len=1200,
)

USE_Q_ENTITY = False  # Set True to supply the dataset's question entity.

instruction = """
Delegate immediately to exactly one graph child. Use q_entity when supplied; otherwise resolve
the starting entity from the question against actual graph labels. Perform question-guided
multi-hop NetworkX traversal up to max_graph_hops, preserve complete paths, print no
neighborhoods, and return at most 20 edges with GRAPH_FINAL. Join paths only inside the root
RLM. If no
supported answer is present, return a FINAL dict with answer_entities=[],
status='insufficient_data', and message='Cannot answer: insufficient data within the graph
traversal limit.'
""".strip()

In [35]:
example = test_rows[41]

In [38]:
print(example["question"])

what county is kansas city kansas


In [37]:
run_context = {"question": example["question"]}
if USE_Q_ENTITY:
    run_context["q_entity"] = [str(entity) for entity in example["q_entity"]]

result = fast_rlm.run(
    run_context,
    graph=indexed_graph,
    config=config,
    output_schema=WebQSPAnswer,
    evaluation_q_entity=example["q_entity"], 
    instruction=instruction,
    # verbosity="silent",
)

with open(result["log_file"]) as log_handle:
    events = [json.loads(line) for line in log_handle if line.strip()]
audit_events = [event for event in events if event.get("event_type", "").startswith("graph_")]
model_calls = sum(
    event.get("usage", {}).get("total_tokens", 0) > 0
    for event in events
)

print("Question: ", row["question"])
print("Prediction:", result["results"]["answer_entities"])
print("Status:", result["results"]["status"])
print("Message:", result["results"]["message"])
print("Golden answer:", [str(answer) for answer in example["answer"]])
print("Model calls:", model_calls)
print("Total tokens:", result["usage"]["total_tokens"])
print("Graph-native audit:")
for event in audit_events:
    if event["event_type"] == "graph_seed_resolution":
        print(
            f"  graph_seed_resolution: mentions={event.get('entity_mentions')}, "
            f"candidates={event.get('candidates')}, source={event.get('seed_source')}, "
            f"count={event.get('seed_count')}, seeds={event.get('seeds')}, "
            f"min_confidence={event.get('min_entity_confidence')}, "
            f"methods={event.get('seed_match_methods')}, "
            f"precision={event.get('entity_precision')}, recall={event.get('entity_recall')}"
        )
        continue
    details = event.get("graph", {})
    operations = event.get("operations")
    suffix = f", operations={operations}" if operations is not None else ""
    hops = event.get("max_graph_hops")
    suffix += f", max_graph_hops={hops}" if hops is not None else ""
    seed_source = event.get("seed_source")
    if seed_source is not None:
        suffix += f", seeds={seed_source}:{event.get('seed_count')}"
        suffix += f", match_methods={event.get('seed_match_methods')}"
    print(
        f"  {event['event_type']}: {details.get('graph_type')} "
        f"nodes={details.get('nodes')}, edges={details.get('edges')}{suffix}"
    )

📝 Logging to: /Users/kapish/Kapish/RP/fast-rlm/examples/logs/run_2026-08-21T12-16-12-047Z.jsonl

✔ Python Ready
✔ requests + httpx ready (env setup took 2512ms)


⠋ Generating code...
✔ Code generated
⠋ Generating code...


────────────────────────────── Depth 0 · Step 1/4 ──────────────────────────────
╭ Reasoning ───────────────────────────────────────────────────────────────────╮
│ The user wants to know what county Kansas City, Kansas is in. I need to      │
│ delegate to a graph query to find this information. Let me use the           │
│ graph_query function with the question and the specified parameters.```      │
╰─────────────────────────────────────────────────────────────────��────────────╯
No code generated. Stopping.
╭ Usage ───────────────────────────────────────────────────────────────────────╮
│ Step:  1,351 prompt, 109 completion | Cost: Unknown                          │
│ Total: 1,351 prompt, 109 completion | Cost: Unknown                          │
╰──────────────────────────────────────────────────────────────────────────────╯


✔ Code generated
⠋ Generating code...


────────────────────────────── Depth 0 · Step 2/4 ──────────────────────────────
╭ Reasoning ───────────────────────────────────────────────────────────────────╮
│ The user wants me to delegate immediately with graph_query. Let me provide   │
│ the proper repl block.```                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯
No code generated. Stopping.
╭ Usage ───────────────────────────────────────────────────────────────────────╮
│ Step:  1,441 prompt, 86 completion | Cost: Unknown                           │
│ Total: 2,792 prompt, 195 completion | Cost: Unknown                          │
╰──────────────────────────────────────────────────────────────────────────────╯


...

↳ graph_query called
✔ Python Ready
✔ requests + httpx ready (env setup took 2550ms)


⠋ Generating code...
✔ Code generated


    │ ─────────────────────────── Depth 1 · Step 1/4 ───────────────────────────
    │ ╭ Reasoning ─────────────────────────────────────────────────────────────╮
    │ │ The question is "what county is kansas city kansas". I need to find    │
    │ │ the county that Kansas City, Kansas is in. The entity is "kansas city  │
    │ │ kansas" and I need relation terms that would connect a city to its     │
    │ │ county.                                                                │
    │ │                                                                        │
    │ │ Good relation terms would be:                                          │
    │ │ - "county" - direct relation                                           │
    │ │ - "located_in_county"                                                  │
    │ │ - "county_seat"                                                        │
    │ │ - "in_county"                                                          │
    │ │ - "administrative_co

⠋ Generating code...
✔ Code generated


────────────────────────────── Depth 0 · Step 4/4 ──────────────────────────────
╭ Reasoning ───────────────────────────────────────────────────────────────────╮
│ The subgraph shows that Kansas City (Kansas) has the county relationship to  │
│ Wyandotte County. The reverse edge confirms this. The answer is Wyandotte    │
│ County.                                                                      │
╰─────────────────────────────────────────────────────────────────��────────────╯
╭ Python Code ─────────────────────────────────────────────────────────────────╮
│ 1 FINAL({"answer_entities": ["Wyandotte County"], "status": "answered",      │
│ "message": "Kansas City, Kansas is located in Wyandotte County, confirmed by │
│ the county relation."})                                                      │
╰──────────────────────────────────────────────────────────────────────────────╯
╭ Result ──────────────────────────────────────────────────────────────────────╮
│ [EMPTY OUTPUT]           

In [8]:
BATCH_LIMIT = 10

In [20]:
rows = test_rows.select(range(BATCH_LIMIT))

In [21]:
print(len(rows))

10


In [25]:
for index,row in enumerate(rows) :
    print(f'################## Example {index} ##################')
    run_context = {"question": row["question"]}
    if USE_Q_ENTITY:
        run_context["q_entity"] = [str(entity) for entity in row["q_entity"]]
    
    result = fast_rlm.run(
        run_context,
        graph=indexed_graph,
        config=config,
        output_schema=WebQSPAnswer,
        evaluation_q_entity=row["q_entity"], 
        instruction=instruction,
        verbosity="silent",
    )
    
    with open(result["log_file"]) as log_handle:
        events = [json.loads(line) for line in log_handle if line.strip()]
    audit_events = [event for event in events if event.get("event_type", "").startswith("graph_")]
    model_calls = sum(
        event.get("usage", {}).get("total_tokens", 0) > 0
        for event in events
    )
    print("Question: ", row["question"])
    print("Prediction:", result["results"]["answer_entities"])
    print("Status:", result["results"]["status"])
    print("Message:", result["results"]["message"])
    print("Golden answer:", [str(answer) for answer in row["answer"]])
    print("Model calls:", model_calls)
    print("Total tokens:", result["usage"]["total_tokens"])
    print("Graph-native audit:")
    for event in audit_events:
        if event["event_type"] == "graph_seed_resolution":
            print(
                f"  graph_seed_resolution: mentions={event.get('entity_mentions')}, "
                f"candidates={event.get('candidates')}, source={event.get('seed_source')}, "
                f"count={event.get('seed_count')}, seeds={event.get('seeds')}, "
                f"min_confidence={event.get('min_entity_confidence')}, "
                f"methods={event.get('seed_match_methods')}, "
                f"precision={event.get('entity_precision')}, recall={event.get('entity_recall')}"
            )
            continue
        details = event.get("graph", {})
        operations = event.get("operations")
        suffix = f", operations={operations}" if operations is not None else ""
        hops = event.get("max_graph_hops")
        suffix += f", max_graph_hops={hops}" if hops is not None else ""
        seed_source = event.get("seed_source")
        if seed_source is not None:
            suffix += f", seeds={seed_source}:{event.get('seed_count')}"
            suffix += f", match_methods={event.get('seed_match_methods')}"
        print(
            f"  {event['event_type']}: {details.get('graph_type')} "
            f"nodes={details.get('nodes')}, edges={details.get('edges')}{suffix}"
        )

################## Example 0 ##################
Question what does jamaican people speak
Prediction: ['Jamaican Creole English Language']
Status: answered
Message: Jamaica's people speak Jamaican Creole English Language (path: Jamaica -[languages_spoken]-> Jamaican Creole English Language).
Golden answer: ['Jamaican English', 'Jamaican Creole English Language']
Model calls: 3
Total tokens: 4271
Graph-native audit:
  graph_channel_open: MultiDiGraph nodes=62813, edges=195828, max_graph_hops=4
  graph_query_start: MultiDiGraph nodes=62813, edges=195828, max_graph_hops=4
  graph_seed_resolution: mentions=['jamaican people', 'jamaica'], candidates=[{'node': 'Jamaica', 'score': 1, 'method': 'exact_label', 'mention_coverage': 1}, {'node': 'Jamaica', 'score': 1, 'method': 'exact_label', 'mention_coverage': 1}, {'node': 'Jamaican English', 'score': 0.7929, 'method': 'trigram_prefix', 'mention_coverage': 1}, {'node': 'Air Jamaica', 'score': 0.6893, 'method': 'partial_trigram', 'mention_coverage

In [28]:
for line in rows[8]["graph"] :
    print(line,"\n")

['Eleanor Roosevelt', 'people.deceased_person.place_of_death', 'Manhattan'] 

['Assassination of John F. Kennedy', 'book.book_subject.works', 'Mortal Error: The Shot That Killed JFK'] 

['John F. Kennedy Jr.', 'base.popstra.celebrity.friendship', 'm.0645k94'] 

['Philip Johnson', 'freebase.valuenotation.is_reviewed', 'Gender'] 

['The Good Shepherd', 'film.film.language', 'Spanish Language'] 

['Cyrus Vance', 'people.deceased_person.place_of_burial', 'Arlington National Cemetery'] 

['John F. Kennedy Catholic High School', 'education.educational_institution.colors', 'Red'] 

['m.0b48tfl', 'common.webpage.resource', 'American Moments with Walter Cronkite'] 

['United States of America', 'location.location.time_zones', 'Pacific Time Zone'] 

['Cape Canaveral Air Force Station', 'location.location.containedby', 'Brevard County'] 

['President of the United States', 'freebase.equivalent_topic.equivalent_type', 'US President'] 

['m.02_97rn', 'celebrities.romantic_relationship.relationship_